# 🔋 Purdue Qiskit Fall Fest — Tutorial
## Molecular VQE Walkthrough: Finding Ground States & Potential Energy Surfaces

Welcome to the tutorial! Even if this is your very first time using Qiskit or quantum computing, this notebook will walk you step-by-step through the core algorithm used in quantum chemistry: the **Variational Quantum Eigensolver (VQE)**.

---

### 💡 The Big Picture: What is VQE?
In quantum chemistry and materials science, molecules naturally relax into their lowest-energy configuration—called the **ground state**. If you know the ground state energy of a chemical system, you can predict:
* Whether a chemical reaction will happen spontaneously.
* How strong an atomic bond is.
* The operating voltage of a battery cell.

**How VQE Works in 4 Steps:**
1. **The Hamiltonian ($H$):** We represent the molecule's electrons and nuclear repulsion as a sum of Pauli matrix strings (like $Z \otimes Z$ or $X \otimes X$).
2. **The Quantum Ansatz ($|\psi(\vec{\theta})\rangle$):** We build a parameterized quantum circuit with tunable rotation angles $\vec{\theta}$ (like knobs on a radio).
3. **The Quantum Measurement (Estimator):** We execute the circuit on a quantum computer or simulator to measure the expected energy: $E(\vec{\theta}) = \langle \psi(\vec{\theta}) | H | \psi(\vec{\theta}) \rangle$.
4. **The Classical Optimizer:** A classical optimization algorithm (like `SLSQP` or `COBYLA`) adjusts the rotation angles $\vec{\theta}$ step-by-step until it finds the minimum possible energy!


In [ ]:
# Step 1: Import core dependencies
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Qiskit 1.x components
from qiskit.circuit.library import EfficientSU2
from qiskit.primitives import StatevectorEstimator

# Pre-packaged hackathon utilities
from utils.battery_helpers import (
    load_benchmarks,
    get_h2_hamiltonian,
    plot_pes_comparison
)

print("✅ Qiskit and helper modules successfully imported!")


### Step 2: Loading the Molecular Hydrogen ($H_2$) Hamiltonian
At the equilibrium bond distance $R = 0.735Å$ (the natural bond length of $H_2$), we can represent the 2 electrons of molecular hydrogen using a 2-qubit Hamiltonian:
$$H = g_0 (I \otimes I) + g_1 (I \otimes Z) + g_2 (Z \otimes I) + g_3 (Z \otimes Z) + g_4 (X \otimes X)$$

Notice that $H$ is made of Pauli operators ($I, X, Z$) multiplied by real numerical weights (the $g_i$ coefficients).


In [ ]:
r_eq = 0.735  # Equilibrium bond distance in Angstroms
H = get_h2_hamiltonian(r_eq)

print(f"Qubit Hamiltonian for H2 at R = {r_eq} Å:")
print(H)


### Step 3: Creating the Parameterized Quantum Circuit (Ansatz)
We use `EfficientSU2`, a standard hardware-efficient ansatz. It consists of:
* Parameterized single-qubit rotations ($R_y, R_z$) on each qubit.
* Entangling two-qubit CNOT gates to allow electrons to interact quantum-mechanically.


In [ ]:
# 2 qubits, 1 repetition layer, linear entanglement
ansatz = EfficientSU2(num_qubits=2, reps=1, entanglement='linear')

print("Number of tunable angle parameters:", ansatz.num_parameters)
ansatz.decompose().draw('mpl')


### Step 4: Setting up VQE with Qiskit's `StatevectorEstimator`
In Qiskit 1.x, expectation values are computed using the **Estimator** primitive. We define a Python cost function that takes parameter angles $\vec{\theta}$ and returns the expected energy.


In [ ]:
estimator = StatevectorEstimator()
loss_tracker = []

def vqe_cost_function(params):
    # Pass (circuit, observable, parameter_values) to Estimator
    job = estimator.run([(ansatz, H, params)])
    result = job.result()
    energy = float(result[0].data.evs)
    loss_tracker.append(energy)
    return energy

# Test with random initial angles
np.random.seed(42)
initial_angles = np.random.uniform(0, 2 * np.pi, ansatz.num_parameters)
test_energy = vqe_cost_function(initial_angles)
print(f"Initial (unoptimized) energy: {test_energy:.4f} Hartree")


### Step 5: Minimizing the Energy with Classical Optimization
We let SciPy's classical optimizer (`SLSQP`) search for the parameter angles that minimize our cost function.


In [ ]:
# Run classical optimization loop
res = minimize(vqe_cost_function, initial_angles, method='SLSQP', options={'maxiter': 50, 'ftol': 1e-6})

print("Optimization converged?", res.success)
print(f"Calculated VQE Ground State Energy: {res.fun:.5f} Hartree")

# Compare with exact classical Full Configuration Interaction (FCI)
benchmarks = load_benchmarks()
fci_energy = benchmarks['H2_dissociation']['equilibrium_FCI_Hartree']
error_mha = abs(res.fun - fci_energy) * 1000

print(f"Exact Classical FCI Energy:         {fci_energy:.5f} Hartree")
print(f"Discrepancy:                        {error_mha:.3f} mHa")
if error_mha <= 1.6:
    print("🎉 Chemical accuracy reached! (Within ±1.6 mHa / 1 kcal/mol of exact nature)")


### Step 6: Visualizing the Optimization Trajectory
Let's plot how the optimizer tuned the quantum circuit angles to descend into the ground state:


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(loss_tracker, 'o-', color='#1f77b4', label='VQE Iterations')
plt.axhline(fci_energy, color='red', linestyle='--', label=f'Exact FCI ({fci_energy:.4f} Ha)')
plt.xlabel("Optimizer Iterations", fontsize=11)
plt.ylabel("Energy (Hartree)", fontsize=11)
plt.title("VQE Optimization Trajectory for H2", fontsize=13, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


---
### Key Takeaways Before Starting the Challenge:
1. **$H_2$ Hamiltonian:** Gives the total energy (electron-electron, electron-nuclear, and nuclear-nuclear repulsion).
2. **The Ground State:** The lowest point of the energy curve corresponds to the stable molecule.
3. **Next Step:** Open **`02_challenge_submission.ipynb`** to sweep the bond length, compute the battery voltage for Lithium-Graphite, and discover a next-gen battery material!
